# 06 — Analyse des erreurs et interprétabilité

Ce notebook analyse les erreurs du modèle :
- Faux négatifs (glaucomes manqués — cliniquement dangereux)
- Faux positifs (sains classés malades)
- Feature importance (RF / XGBoost)
- SHAP values pour l'interprétabilité
- Learning curves

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.config import PROJECT_ROOT, FEATURE_CACHE_DIR, RANDOM_STATE
from src.data_loading import get_ml_subset_df, get_train_val_test_splits
from src.feature_extraction import build_feature_matrix
from src.pipeline_ml import get_classifier, create_pipeline, fit_scaler, threshold_predict, find_threshold_for_specificity
from src.evaluation_metrics import (
    compute_all_metrics, compute_all_metrics_with_ci,
    plot_roc_comparison, plot_confusion_matrix,
)
from src.preprocessing import load_image

print('Modules chargés.')

## 1. Chargement des données et du modèle

In [ ]:
# Charger les données du dossier 0
df = get_ml_subset_df()
train_df, val_df, test_df = get_train_val_test_splits(df)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'RG dans test: {(test_df["class"] == "RG").sum()}')

In [ ]:
# Extraire les features (utilise le cache si disponible)
FEATURE_TYPE = 'full'  # Changer selon le type à analyser

X_train, y_train = build_feature_matrix(
    train_df, feature_type=FEATURE_TYPE, preprocess=True,
    cache_dir=str(FEATURE_CACHE_DIR)
)
X_val, y_val = build_feature_matrix(
    val_df, feature_type=FEATURE_TYPE, preprocess=True,
    cache_dir=str(FEATURE_CACHE_DIR)
)
X_test, y_test = build_feature_matrix(
    test_df, feature_type=FEATURE_TYPE, preprocess=True,
    cache_dir=str(FEATURE_CACHE_DIR)
)

print(f'X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')

In [ ]:
# Entraîner le modèle (ou charger depuis models/)
from src.pipeline_ml import get_smote

scaler = fit_scaler(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

X_train_r, y_train_r = get_smote(X_train_s, y_train)

clf = get_classifier('xgb', scale_pos_weight=29)
clf.fit(X_train_r, y_train_r)
print('Modèle entraîné.')

## 2. Analyse des erreurs

In [ ]:
# Prédictions
y_proba_test = clf.predict_proba(X_test_s)
y_score_test = y_proba_test[:, 1]

# Seuil optimisé pour 95% spécificité
y_proba_val = clf.predict_proba(X_val_s)
threshold = find_threshold_for_specificity(y_val, y_proba_val[:, 1], target_spec=0.95)
y_pred_test = threshold_predict(y_proba_test, threshold)

print(f'Seuil optimisé: {threshold:.4f}')
print(f'\nMétriques sur le test set:')
metrics = compute_all_metrics(y_test, y_score_test)
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Identifier les erreurs
test_df_indexed = test_df.reset_index(drop=True)

fn_mask = (y_test == 1) & (y_pred_test == 0)  # Faux négatifs (glaucomes manqués)
fp_mask = (y_test == 0) & (y_pred_test == 1)  # Faux positifs
tp_mask = (y_test == 1) & (y_pred_test == 1)  # Vrais positifs
tn_mask = (y_test == 0) & (y_pred_test == 0)  # Vrais négatifs

print(f'Faux négatifs (glaucomes manqués): {fn_mask.sum()}')
print(f'Faux positifs: {fp_mask.sum()}')
print(f'Vrais positifs: {tp_mask.sum()}')
print(f'Vrais négatifs: {tn_mask.sum()}')

In [ ]:
# Visualiser les faux négatifs (les plus dangereux cliniquement)
fn_indices = np.where(fn_mask)[0]
n_show = min(8, len(fn_indices))

if n_show > 0:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle('Faux négatifs (glaucomes manqués)', fontsize=14)
    for i, ax in enumerate(axes.flat):
        if i >= n_show:
            ax.axis('off')
            continue
        idx = fn_indices[i]
        path = PROJECT_ROOT / test_df_indexed.iloc[idx]['path']
        try:
            img = load_image(path)
            ax.imshow(img)
            ax.set_title(f'P(RG)={y_score_test[idx]:.3f}', fontsize=10)
        except Exception:
            ax.set_title('Image non trouvée')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Aucun faux négatif (parfait !)')

In [ ]:
# Distribution des probabilités par classe
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_score_test[y_test == 0], bins=50, alpha=0.6, label='NRG (sain)', color='blue')
ax.hist(y_score_test[y_test == 1], bins=50, alpha=0.6, label='RG (glaucome)', color='red')
ax.axvline(x=threshold, color='black', linestyle='--', label=f'Seuil={threshold:.3f}')
ax.set_xlabel('Probabilité prédite (RG)')
ax.set_ylabel('Nombre d\'images')
ax.set_title('Distribution des probabilités par classe')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Feature importance

In [ ]:
# Feature importance du modèle
if hasattr(clf, 'feature_importances_'):
    importances = clf.feature_importances_
    indices = np.argsort(importances)[::-1][:30]  # Top 30
    
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(range(30), importances[indices[::-1]])
    ax.set_yticks(range(30))
    ax.set_yticklabels([f'Feature {i}' for i in indices[::-1]])
    ax.set_xlabel('Importance')
    ax.set_title('Top 30 features les plus importantes')
    plt.tight_layout()
    plt.show()
    
    print('\nTop 10 features:')
    for rank, i in enumerate(indices[:10]):
        print(f'  #{rank+1}: Feature {i} (importance={importances[i]:.4f})')

## 4. SHAP values (interprétabilité)

In [ ]:
try:
    import shap
    
    # TreeExplainer pour RF/XGBoost
    explainer = shap.TreeExplainer(clf)
    
    # Calculer SHAP sur un sous-ensemble du test (pour la vitesse)
    n_shap = min(500, len(X_test_s))
    shap_values = explainer.shap_values(X_test_s[:n_shap])
    
    # Summary plot
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_test_s[:n_shap], max_display=20, show=False)
    plt.title('SHAP Summary — Top 20 features')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('shap non installé. Installer avec: pip install shap')
except Exception as e:
    print(f'Erreur SHAP: {e}')

## 5. Learning curves

In [ ]:
from sklearn.model_selection import learning_curve, StratifiedKFold

# Learning curve : performance vs taille du train set
train_sizes, train_scores, val_scores = learning_curve(
    clf, X_train_s, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc',
    n_jobs=-1,
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train AUC', color='blue')
ax.fill_between(train_sizes,
                train_scores.mean(axis=1) - train_scores.std(axis=1),
                train_scores.mean(axis=1) + train_scores.std(axis=1),
                alpha=0.1, color='blue')
ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', label='Val AUC', color='orange')
ax.fill_between(train_sizes,
                val_scores.mean(axis=1) - val_scores.std(axis=1),
                val_scores.mean(axis=1) + val_scores.std(axis=1),
                alpha=0.1, color='orange')
ax.set_xlabel('Taille du training set')
ax.set_ylabel('AUC-ROC')
ax.set_title('Learning Curve — Le modèle bénéficierait-il de plus de données ?')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Matrice de confusion et ROC

In [ ]:
# Matrice de confusion
plot_confusion_matrix(y_test, y_pred_test)
plt.show()

# Métriques avec intervalles de confiance
print('\nMétriques avec IC 95% (bootstrap):')
metrics_ci = compute_all_metrics_with_ci(y_test, y_score_test, n_bootstrap=1000)